# Atelier Preparation de Donnees Textuelles

## Partie 1 - Exploration du corpus

### 1) Chargement des donnees CSV

In [ ]:
import pandas as pd

df = pd.read_csv("../data/smart_reviews_raw.csv", encoding="utf-8")
df.head()

### 2) et 3) Nombre d'avis et nombre de colonnes

In [ ]:
n_avis, n_colonnes = df.shape
print(f"Nombre d'avis : {n_avis}")
print(f"Nombre de colonnes : {n_colonnes}")

### 4) Type de chaque colonne

In [ ]:
df.dtypes

### 5) Valeurs manquantes

In [ ]:
df.isna().sum()

### 6) Identifier quelques types de texte

In [ ]:
import re

texte = df["texte"]

exemples = {
    "normal": texte[texte.str.len().between(30, 60) & texte.notna()].iloc[0],
    "vide": texte[texte.isna()].index[0] if texte.isna().any() else None,
    "url": texte[texte.str.contains("http", na=False)].iloc[0],
    "mention": texte[texte.str.contains("@", na=False)].iloc[0],
    "hashtag": texte[texte.str.contains("#", na=False)].iloc[0],
    "emoji": texte[texte.str.contains(r"[\U0001F300-\U0001FAFF]", na=False, regex=True)].iloc[0],
    "ponctuation": texte[texte.str.contains(r"[!?]{2,}", na=False, regex=True)].iloc[0],
    "majuscules": texte[texte.str.isupper().fillna(False)].iloc[0],
    "repetition": texte[texte.str.contains(r"(.)\1{2,}", na=False, regex=True)].iloc[0],
}

for type_texte, exemple in exemples.items():
    print(f"{type_texte:12s}: {exemple}")

### 7) Longueur des textes

In [ ]:
df["longueur"] = df["texte"].str.len()

stats_longueur = df["longueur"].describe()
print(f"Longueur minimale : {df['longueur'].min()}")
print(f"Longueur maximale : {df['longueur'].max()}")
print(f"Longueur moyenne  : {df['longueur'].mean():.2f}")
print(f"Longueur mediane  : {df['longueur'].median()}")
print(f"Q1 : {df['longueur'].quantile(0.25)}")
print(f"Q3 : {df['longueur'].quantile(0.75)}")
stats_longueur

### 8) Visualisation de la distribution de la longueur des avis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["longueur"].dropna(), bins=40, color="steelblue")
axes[0].set_title("Distribution de la longueur des avis")
axes[0].set_xlabel("Longueur (caracteres)")
axes[0].set_ylabel("Frequence")

axes[1].boxplot(df["longueur"].dropna(), vert=True)
axes[1].set_title("Boxplot de la longueur des avis")
axes[1].set_ylabel("Longueur (caracteres)")

plt.tight_layout()
plt.show()

**a) Existe-t-il des textes anormalement longs ?**

Oui. La moyenne se situe autour de 49 caracteres et le 3e quartile autour de 56, mais le maximum atteint plus de 600 caracteres. Ces valeurs tres eloignees de la distribution principale (visibles comme points isoles au-dessus de la moustache superieure du boxplot) constituent des outliers a examiner : ils peuvent correspondre a des avis tres detailles, du texte duplique/spam, ou du contenu concatene par erreur lors de la collecte.

**b) Existe-t-il beaucoup de textes tres courts ?**

Oui, une partie non negligeable des avis a une longueur proche du minimum observe (2 caracteres), ce qui correspond a des textes quasi vides ou peu informatifs (ex: un seul mot, une ponctuation). Ces textes tres courts risquent d'apporter peu de signal pour la classification de sentiment et devront etre surveilles lors du nettoyage.

### 9) Detection des textes vides

In [ ]:
textes_vides = df[df["texte"].isna() | (df["texte"].str.strip() == "")]
print(f"Nombre de textes vides : {len(textes_vides)}")
textes_vides[["id_avis", "texte"]]

### 10) Detection des doublons sur le texte

In [ ]:
doublons = df[df.duplicated(subset="texte", keep=False) & df["texte"].notna()]
print(f"Nombre de lignes impliquees dans un doublon de texte : {len(doublons)}")
print(f"Nombre de doublons (hors 1re occurrence) : {df['texte'].duplicated().sum()}")
doublons.sort_values("texte")[["id_avis", "texte", "source"]].head(10)

### 11) et 12) Equilibre des classes et classe majoritaire

In [ ]:
sentiment_counts = df["sentiment"].value_counts()
sentiment_pct = df["sentiment"].value_counts(normalize=True) * 100

print(sentiment_counts)
print()
print(sentiment_pct.round(1))

fig, ax = plt.subplots(figsize=(5, 4))
sentiment_counts.plot(kind="bar", color="steelblue", ax=ax)
ax.set_title("Repartition des classes de sentiment")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Nombre d'avis")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"Classe majoritaire : {sentiment_counts.idxmax()} ({sentiment_counts.max()} avis)")

Les classes ne sont **pas equilibrees** : la classe `positif` domine largement (~65% des avis), suivie de `neutre` (~20%) puis `negatif` (~15%). Ce dessequilibre doit etre pris en compte lors de l'entrainement d'un modele (risque de biais vers la classe majoritaire) et lors du choix des metriques d'evaluation.

### 13) Metriques de classification a envisager

Le dataset etant desequilibre (classe `positif` largement majoritaire), l'**accuracy seule** serait trompeuse : un modele qui predirait toujours `positif` obtiendrait deja un score eleve sans etre utile.

Metriques a privilegier :
- **F1-score macro** : moyenne non ponderee du F1 par classe, donne le meme poids a chaque classe (positif, neutre, negatif) quelle que soit sa frequence.
- **F1-score pondere (weighted)** : tient compte du poids reel de chaque classe, utile pour une vue d'ensemble realiste.
- **Precision et rappel par classe** : essentiels pour verifier que la classe minoritaire (`negatif`) n'est pas sacrifiee.
- **Matrice de confusion** : permet de visualiser les confusions entre classes (ex: neutre confondu avec positif).

L'accuracy peut etre suivie en complement mais ne doit pas etre la metrique de decision principale.

### 14) Detection des caracteres particuliers

In [ ]:
patterns = {
    "emojis": r"[\U0001F300-\U0001FAFF]",
    "urls": r"https?://\S+|www\.\S+",
    "hashtags": r"#\w+",
    "mentions": r"@\w+",
    "chiffres": r"\d",
    "caracteres_speciaux": r"[^\w\s]",
    "ponctuation_repetee": r"[!?.]{2,}",
}

texte = df["texte"]
resultats = {
    nom: texte.str.contains(regex, na=False, regex=True).sum()
    for nom, regex in patterns.items()
}

for nom, count in resultats.items():
    print(f"{nom:22s}: {count} textes ({count / len(texte) * 100:.1f}%)")

### 15) Nombre d'avis par source

In [ ]:
source_counts = df["source"].value_counts()
print(source_counts)

fig, ax = plt.subplots(figsize=(6, 4))
source_counts.plot(kind="bar", color="darkorange", ax=ax)
ax.set_title("Nombre d'avis par source")
ax.set_xlabel("Source")
ax.set_ylabel("Nombre d'avis")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Les avis sont globalement bien repartis entre les 5 sources attendues (`web`, `mobile`, `sav`, `email`, `reseaux_sociaux`), chacune representant environ 20% du dataset. On remarque toutefois une **incoherence de casse** : la valeur `WEB` apparait une fois en plus de `web`, ce qui cree artificiellement une 6e categorie. Cette incoherence devra etre harmonisee (mise en minuscules) lors du nettoyage pour ne pas fausser les analyses par source.

### 16) Nombre d'avis par produit

In [ ]:
produit_counts = df["produit"].value_counts()
print(produit_counts)

fig, ax = plt.subplots(figsize=(7, 4))
produit_counts.plot(kind="bar", color="seagreen", ax=ax)
ax.set_title("Nombre d'avis par produit")
ax.set_xlabel("Produit")
ax.set_ylabel("Nombre d'avis")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Les 6 produits (Ordinateur NovaBook, SmartPhone X, SmartPhone Y, Ecouteurs AirSound, SmartWatch Pro, Tablette TabPlus) recoivent un nombre d'avis relativement homogene, entre ~180 et ~220 avis chacun. Il n'y a pas de produit sous-represente de maniere critique, ce qui permettra une analyse par produit fiable sans necessiter de reequilibrage specifique sur cet axe.

## Partie 2 - Nettoyage

### 1) Suppression des lignes avec texte manquant

In [ ]:
print(f"Nombre de lignes avant : {len(df)}")

df_clean = df.dropna(subset=["texte"]).copy()

print(f"Nombre de lignes apres : {len(df_clean)}")
print(f"Lignes supprimees : {len(df) - len(df_clean)}")

### 2) Suppression des doublons

In [ ]:
n_avant = len(df_clean)
print(f"Nombre de lignes avant : {n_avant}")

df_clean = df_clean.drop_duplicates(subset="texte", keep="first").copy()

n_apres = len(df_clean)
print(f"Nombre de lignes apres : {n_apres}")
print(f"Doublons supprimes : {n_avant - n_apres}")

### 3) Suppression des URLs

In [ ]:
def supprimer_urls(texte):
    return re.sub(r"https?://\S+|www\.\S+", "", texte)

exemple_avant = df_clean[df_clean["texte"].str.contains("http", na=False)]["texte"].iloc[0]
exemple_apres = supprimer_urls(exemple_avant)

print(f"Avant : {exemple_avant}")
print(f"Apres : {exemple_apres}")

### 4) Traitement des mentions et hashtags

**Strategie retenue :**
- **Mentions (`@utilisateur`)** : suppression complete. Un nom d'utilisateur ou de marque mentionne n'apporte aucun signal de sentiment et introduit du bruit (identifiants uniques, peu generalisables).
- **Hashtags (`#mot`)** : on **conserve le mot** en supprimant uniquement le symbole `#`. En effet, un hashtag comme `#decu` ou `#genial` peut porter une charge emotionnelle forte et utile pour la classification de sentiment ; le supprimer entierement ferait perdre de l'information pertinente.

In [ ]:
def traiter_mentions_hashtags(texte):
    texte = re.sub(r"@\w+", "", texte)
    texte = re.sub(r"#(\w+)", r"\1", texte)
    return texte

exemple_mention = df_clean[df_clean["texte"].str.contains("@", na=False)]["texte"].iloc[0]
exemple_hashtag = df_clean[df_clean["texte"].str.contains("#", na=False)]["texte"].iloc[0]

print(f"Avant (mention) : {exemple_mention}")
print(f"Apres (mention) : {traiter_mentions_hashtags(exemple_mention)}")
print()
print(f"Avant (hashtag) : {exemple_hashtag}")
print(f"Apres (hashtag) : {traiter_mentions_hashtags(exemple_hashtag)}")

### 5) Nettoyage des espaces

In [ ]:
def nettoyer_espaces(texte):
    return re.sub(r"\s+", " ", texte).strip()

exemple_espaces = df_clean[df_clean["texte"].str.contains(r"  ", na=False, regex=True)]["texte"].iloc[0]

print(f"Avant : {exemple_espaces!r}")
print(f"Apres : {nettoyer_espaces(exemple_espaces)!r}")